# RAG: Baseline Bi-Encoder Retrieval vs. Cross-Encoder Re-Ranking

**Document:** *FIDE Laws of Chess* (official text, in force from 1 January 2023), a 45-page rulebook full of
exceptions ("except where…", "however…"), prohibitions ("it is forbidden…") and appendix overrides
(rapid / blitz rules that change the standard penalties).

**What this notebook does**

| Step | Pipeline 1: Baseline | Pipeline 2: Re-Ranking |
|---|---|---|
| Retrieve | Bi-encoder (`all-MiniLM-L6-v2`) + FAISS inner product → Top 10 | *same Top 10 candidates* |
| Select | Top 3 by cosine similarity | Cross-encoder (`ms-marco-MiniLM-L-6-v2`) scores all 10 (query, chunk) pairs → re-sort → Top 3 |
| Generate | Local LLM (`Qwen2.5-3B-Instruct`) → **Answer 1** | Same LLM, identical prompt → **Answer 2** |

Sections:
1. Setup & configuration
2. Load and chunk the PDF
3. Pipeline 1: bi-encoder index + baseline retrieval
4. Pipeline 2: cross-encoder re-ranking
5. LLM + shared prompt
6. Failure-mode analysis (stress tests): rank shifts + Answer 1 vs Answer 2
7. Aggregate retrieval evaluation over a labelled query set
8. Latency profiling
9. Scaling experiment (why not cross-encode everything?)
10. Save results

## 1. Setup & configuration

In [1]:
# Install dependencies (uncomment on a fresh environment)
# %pip install pymupdf sentence-transformers faiss-cpu transformers accelerate torch pandas

import os, re, time, json, random, warnings
import numpy as np
import pandas as pd
import fitz                      # PyMuPDF, for PDF text extraction
import faiss                     # vector store (exact inner-product index)
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# ---- Configuration -------------------------------------------------------
PDF_PATH        = "data/FIDE_Laws_of_Chess_2023.pdf"
BI_ENCODER      = "sentence-transformers/all-MiniLM-L6-v2"   # dense bi-encoder
CROSS_ENCODER   = "cross-encoder/ms-marco-MiniLM-L-6-v2"     # re-ranker
LLM_NAME        = "Qwen/Qwen2.5-3B-Instruct"                 # local open-weights LLM
CHUNK_MAX_CHARS = 700     # target chunk size (characters)
TOP_K_RETRIEVE  = 10      # candidates from the bi-encoder
TOP_K_CONTEXT   = 3       # chunks passed to the LLM
MAX_NEW_TOKENS  = 220
SEED            = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("results", exist_ok=True)
print("torch", torch.__version__, "| device:", DEVICE)

C:\Users\aksha\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1017 01:29:13.989000 14472 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


torch 2.10.0+cpu | device: cpu


## 2. Load and chunk the PDF

The Laws are numbered hierarchically (`3.8.2.2`, `A.5.2`, `III.4` …), so I chunk on **clause boundaries**
instead of fixed windows:

1. Extract text with PyMuPDF.
2. Track the current section heading (`Article 9: The Drawn Game`, `Appendix A. Rapid Chess`, …).
3. Start a new *clause* whenever a line begins with a clause number; continuation lines are appended.
4. Greedily pack consecutive clauses of the **same section** into chunks of ≤ ~700 characters
   (a chunk never crosses an Article/Appendix boundary).
5. Oversized blocks (the glossary table) are hard-split on word boundaries.
6. Each chunk is prefixed with its section heading, so the encoder sees where the clause comes from.

Every chunk keeps the list of clause numbers it contains; this lets me check retrieval against
**gold clause labels** later.

In [2]:
SECTION_RE = re.compile(
    r"^(Article \d+:.*|Appendix [A-D]\..*|Guidelines [IV]+\..*|Glossary of Terms.*"
    r"|BASIC RULES OF PLAY|COMPETITIVE RULES OF PLAY|FIDE LAWS OF CHESS)\s*$")
# A clause number such as 9.6.2, A.5.1.2 or III.4 at the start of a line
# (the negative look-ahead skips wrapped cross-references like "4.1 – 4.7").
CLAUSE_RE = re.compile(r"^((?:\d+|[A-D]|I{1,3}|II\.\d+)(?:\.\d+)+\.?)[\s ]+(?![–-])\S")


def load_pdf_lines(path):
    doc = fitz.open(path)
    text = "\n".join(page.get_text() for page in doc)
    print(f"Pages: {doc.page_count} | characters: {len(text):,}")
    return [line.replace(" ", " ").rstrip() for line in text.split("\n")]


def split_into_clauses(lines):
    # Walk the lines, tracking the section heading and starting a new clause at each clause number.
    clauses, section, cur = [], "Preamble", None
    for line in lines:
        s = line.strip()
        if not s or s.startswith("Source: https"):
            continue
        if SECTION_RE.match(s):                       # new Article / Appendix / Guideline
            if cur: clauses.append(cur); cur = None
            section = s
            continue
        is_glossary = section.startswith("Glossary")
        m = None if is_glossary else CLAUSE_RE.match(s)
        if m:                                         # a new numbered clause starts here
            if cur: clauses.append(cur)
            cur = {"section": section, "clause": m.group(1).rstrip("."), "text": s}
        elif cur is None:                             # un-numbered text at the start of a section
            cur = {"section": section, "clause": "Glossary" if is_glossary else "-", "text": s}
        else:                                         # continuation line of the current clause
            cur["text"] += " " + s
    if cur: clauses.append(cur)
    return clauses


def pack_chunks(clauses, max_chars=CHUNK_MAX_CHARS):
    # Greedily merge consecutive clauses of the same section up to max_chars.
    chunks, buf = [], None
    for c in clauses:
        if buf and buf["section"] == c["section"] and len(buf["text"]) + len(c["text"]) <= max_chars:
            buf["text"] += "\n" + c["text"]
            buf["clauses"].append(c["clause"])
        else:
            if buf: chunks.append(buf)
            buf = {"section": c["section"], "clauses": [c["clause"]], "text": c["text"]}
    chunks.append(buf)

    # Hard-split anything still far too long (the glossary table comes out as one huge block).
    out = []
    for c in chunks:
        if len(c["text"]) <= max_chars * 1.5:
            out.append(c); continue
        piece = ""
        for w in c["text"].split():
            if len(piece) + len(w) + 1 > max_chars:
                out.append({**c, "text": piece.strip()}); piece = ""
            piece += " " + w
        if piece.strip(): out.append({**c, "text": piece.strip()})

    for i, c in enumerate(out):
        c["id"] = i
        c["passage"] = f"[{c['section']}]\n{c['text']}"   # text that is embedded / shown to the LLM
    return out


lines   = load_pdf_lines(PDF_PATH)
clauses = split_into_clauses(lines)
chunks  = pack_chunks(clauses)
passages = [c["passage"] for c in chunks]

lens = [len(p) for p in passages]
print(f"Clauses: {len(clauses)} | Chunks: {len(chunks)} | chars/chunk min={min(lens)} "
      f"median={int(np.median(lens))} max={max(lens)}")
pd.DataFrame([{"id": c["id"], "section": c["section"][:45], "clauses": ", ".join(c["clauses"][:6]),
               "chars": len(c["passage"])} for c in chunks]).iloc[[0, 12, 37, 52, 57, 65, 66, 69]]

Pages: 45 | characters: 84,099
Clauses: 365 | Chunks: 141 | chars/chunk min=131 median=684 max=1033


,id,section,clauses,chars
0,0,FIDE LAWS OF CHESS,"-, 0.1",419
12,12,Article 3: The Moves of the Pieces,"3.8.2.1, 3.8.2.2, 3.9",496
37,37,Article 7: Irregularities,7.5.1,536
52,52,Article 9: The Drawn Game,"9.5.1, 9.5.2, 9.5.3, 9.6, 9.6.1",670
57,57,Article 11: The Conduct of the Players,"11.3.2.1, 11.3.2.2",552
65,65,Appendix A. Rapid Chess,"A.1, A.2, A.3, A.4",717
66,66,Appendix A. Rapid Chess,"A.4.1, A.4.2, A.4.3, A.5, A.5.1, A.5.1.1",684
69,69,Appendix A. Rapid Chess,"A.5.3, A.5.4, A.5.5, A.6",719


In [3]:
# Peek at one chunk to see what the encoders and the LLM receive
print(chunks[12]["passage"])

[Article 3: The Moves of the Pieces]
3.8.2.1    The right to castle has been lost: 1) If the king has already moved, or 2) With a rook that has already moved.
3.8.2.2    Castling is prevented temporarily: 3) If the square on which the king stands, or the square which it must cross, or the square which it is to occupy, is attacked by one or more of the opponent's pieces, or 4) If there is any piece between the king and the rook with which castling is to be effected.
3.9     The king in check:


## 3. Pipeline 1: Baseline retrieval (bi-encoder)

The bi-encoder embeds every chunk **independently** into a 384-d vector. Embeddings are L2-normalised,
so the inner product in `faiss.IndexFlatIP` equals cosine similarity. At query time only the query is
embedded and compared against the stored vectors.

In [4]:
bi_encoder = SentenceTransformer(BI_ENCODER, device=DEVICE)

t0 = time.perf_counter()
chunk_emb = bi_encoder.encode(passages, batch_size=32, normalize_embeddings=True,
                              convert_to_numpy=True).astype("float32")
index = faiss.IndexFlatIP(chunk_emb.shape[1])     # exact inner-product search
index.add(chunk_emb)
print(f"Indexed {index.ntotal} chunks, dim={chunk_emb.shape[1]} in {time.perf_counter()-t0:.2f}s (one-off cost)")


def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    # Pipeline 1, steps a-b: embed the query and return the top-k chunk ids with cosine scores.
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(q, k)
    return ids[0].tolist(), scores[0].tolist()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5671.70it/s]

Indexed 141 chunks, dim=384 in 1.41s (one-off cost)


## 4. Pipeline 2: Cross-encoder re-ranking

The cross-encoder reads **query and chunk together** as one sequence (`[CLS] query [SEP] chunk [SEP]`),
so every query token can attend to every chunk token. It outputs a single relevance logit per pair.
It re-scores only the 10 bi-encoder candidates.

In [5]:
cross_encoder = CrossEncoder(CROSS_ENCODER, device=DEVICE)


def rerank(query, cand_ids):
    # Pipeline 2, steps b-c: score each (query, chunk) pair jointly and sort descending.
    pairs = [(query, passages[i]) for i in cand_ids]
    ce_scores = cross_encoder.predict(pairs, batch_size=16)
    order = np.argsort(-ce_scores)
    return [cand_ids[j] for j in order], [float(ce_scores[j]) for j in order]


def run_both_retrievals(query):
    # Pipeline 1 top-3 and Pipeline 2 top-3 from the SAME top-10 candidate set.
    cand_ids, bi_scores = baseline_retrieve(query)
    rr_ids, ce_scores = rerank(query, cand_ids)
    return {"cand_ids": cand_ids, "bi_scores": bi_scores,
            "rr_ids": rr_ids, "ce_scores": ce_scores,
            "baseline_top3": cand_ids[:TOP_K_CONTEXT], "rerank_top3": rr_ids[:TOP_K_CONTEXT]}

# quick smoke test
r = run_both_retrievals("Can a player castle if the king is in check?")
print("baseline top-3:", [chunks[i]["clauses"][0] for i in r["baseline_top3"]])
print("re-ranked top-3:", [chunks[i]["clauses"][0] for i in r["rerank_top3"]])

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7057.95it/s]

baseline top-3: ['3.8.2.1', '3.9.1', '1.4.1']
re-ranked top-3: ['3.8.2.1', 'A.5.3', '3.9.1']


## 5. LLM and the shared prompt

Both pipelines use **the same local LLM, the same prompt template and greedy decoding**, so any
difference between Answer 1 and Answer 2 comes only from *which 3 chunks* were retrieved.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME, dtype=torch.bfloat16 if DEVICE == "cpu" else torch.float16).to(DEVICE)
llm.eval()

SYSTEM_PROMPT = (
    "You are an assistant that answers questions about the FIDE Laws of Chess. "
    "Answer ONLY using the context passages provided. Quote or cite the article numbers you rely on. "
    "If the context does not contain the answer, say: 'The provided context does not answer this.' "
    "Keep the answer to 2-4 sentences.")


def build_prompt(query, chunk_ids):
    # Identical template for both pipelines; only the context changes.
    context = "\n\n".join(f"[Passage {n}]\n{passages[i]}" for n, i in enumerate(chunk_ids, 1))
    user = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]


@torch.no_grad()
def generate_answer(query, chunk_ids):
    msgs = build_prompt(query, chunk_ids)
    enc = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                        return_tensors="pt", return_dict=True).to(DEVICE)
    t0 = time.perf_counter()
    out = llm.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)  # greedy = deterministic
    dt = time.perf_counter() - t0
    text = tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return text, dt

print(generate_answer("How does the knight move?", baseline_retrieve("How does the knight move?")[0][:3])[0])

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading weights:   5%|▍         | 21/434 [00:00<00:01, 209.51it/s]

Loading weights:  10%|▉         | 42/434 [00:00<00:02, 178.58it/s]

Loading weights:  14%|█▍        | 61/434 [00:00<00:02, 158.54it/s]

Loading weights:  18%|█▊        | 78/434 [00:00<00:02, 137.19it/s]

...

Loading weights:  46%|████▋     | 201/434 [00:03<00:13, 16.84it/s]

Loading weights:  47%|████▋     | 204/434 [00:04<00:15, 14.69it/s]

Loading weights:  48%|████▊     | 208/434 [00:04<00:14, 15.94it/s]

Loading weights: 100%|██████████| 434/434 [00:04<00:00, 99.39it/s]

A knight moves to one of the squares nearest to that on which it stands but not on the same rank, file or diagonal. This movement is described in Appendix C.10.3 of the FIDE Laws of Chess.


## 6. Failure-mode analysis (stress tests)

Each test query has a **gold clause** (the article that actually answers it, checked by reading the Laws).
For every query I report:

* the gold chunk's rank under the bi-encoder (Pipeline 1) and after cross-encoder re-ranking (Pipeline 2),
* the full top-10 table with both scores,
* **Answer 1** (baseline top-3) vs **Answer 2** (re-ranked top-3).

These queries came out of a sweep of ~60 candidate questions (section 7 holds the full set).
I kept the ones where the bi-encoder buries the gold clause at ranks 4–10 and the cross-encoder lifts it into the top 3,
plus two **counter-examples** where re-ranking makes things worse.

In [7]:
FAILURE_QUERIES = [
    {"category": "Keyword trap / distractor",
     "query": "Can a player use two hands to castle?",
     "gold": ["4.1", "7.5.4"],
     "why": "The word 'castle' pulls in the castling-procedure clauses (3.8.x, 4.4.x). The rule is in Art. 4.1 "
            "('one hand only'), and the penalty is in 7.5.4, which never mentions castling as a topic."},
    {"category": "Keyword trap / synonyms",
     "query": "Does a player have to keep writing down moves when very short of time?",
     "gold": ["8.4"],
     "why": "The query says 'writing down' and 'short of time'; Art. 8.4 says 'not obliged to meet the requirements "
            "of Article 8.1.1' and 'less than five minutes'. No shared vocabulary."},
    {"category": "Exception / corner case",
     "query": "In a rapid game, what happens if nobody notices an illegal move?",
     "gold": ["A.5.2"],
     "why": "The general illegal-move rule (7.5.x) dominates; the rapid-chess exception that lets the illegal move "
            "STAND lives in Appendix A.5.2."},
    {"category": "Exception / penalty located elsewhere",
     "query": "What is the penalty for an incorrect draw claim in rapid chess?",
     "gold": ["A.3"],
     "why": "Art. 9.5.3 says the opponent gets two minutes, but Appendix A.3 overrides it to ONE minute in rapid."},
    {"category": "Negation & constraints",
     "query": "Is a game drawn automatically without any claim after a long series of moves with no captures?",
     "gold": ["9.6.2"],
     "why": "'without any claim' is a negation. The bi-encoder favours chunks about claims (50-move CLAIM, 9.3), "
            "while the automatic 75-move rule is 9.6.2."},
    {"category": "Other: vital chunk buried at rank 4-10",
     "query": "Is the time spent before pressing the clock counted against the player?",
     "gold": ["6.2.2"],
     "why": "Generic clock vocabulary is spread across Article 6; the specific sentence is in 6.2.2."},
    # ---- counter-examples (re-ranking hurts) -----------------------------------------------
    {"category": "Counter-example: re-ranking hurts",
     "query": "My opponent ran out of time but I only have a bare king left. Do I win?",
     "gold": ["6.9"],
     "why": "The 'however, drawn if the opponent cannot checkmate' exception in 6.9 is at bi-encoder rank 3, "
            "but the cross-encoder prefers checkmate/winning chunks and pushes it to rank 5."},
    {"category": "Counter-example: re-ranking hurts (negation)",
     "query": "Is the arbiter allowed to tell players how many moves have been made?",
     "gold": ["12.6"],
     "why": "12.6 ('shall not indicate the number of moves completed') is rank 2 for the bi-encoder; the cross-encoder "
            "rewards passages that literally talk about counting moves (7.3, 9.3) and drops it to rank 7."},
]


def gold_rank(id_list, gold):
    # 1-based rank of the first chunk containing any gold clause, or None if absent.
    for r, i in enumerate(id_list, 1):
        if any(g in chunks[i]["clauses"] for g in gold):
            return r
    return None


def rank_table(res, gold):
    # Top-10 table: bi-encoder rank/score vs cross-encoder rank/score for every candidate.
    ce_rank = {cid: r for r, cid in enumerate(res["rr_ids"], 1)}
    ce_score = dict(zip(res["rr_ids"], res["ce_scores"]))
    rows = []
    for r, (cid, s) in enumerate(zip(res["cand_ids"], res["bi_scores"]), 1):
        is_gold = any(g in chunks[cid]["clauses"] for g in gold)
        rows.append({"chunk": cid, "clauses": ", ".join(chunks[cid]["clauses"][:5]),
                     "bi_rank": r, "cosine": round(s, 3), "ce_rank": ce_rank[cid],
                     "ce_score": round(ce_score[cid], 2), "shift": r - ce_rank[cid],
                     "GOLD": "★" if is_gold else ""})
    return pd.DataFrame(rows).sort_values("ce_rank").reset_index(drop=True)

In [8]:
from IPython.display import display, Markdown

failure_results = []
for fq in FAILURE_QUERIES:
    res = run_both_retrievals(fq["query"])
    b_rank, r_rank = gold_rank(res["cand_ids"], fq["gold"]), gold_rank(res["rr_ids"], fq["gold"])
    ans1, t1 = generate_answer(fq["query"], res["baseline_top3"])
    ans2, t2 = generate_answer(fq["query"], res["rerank_top3"])
    gold_chunk = next(i for i in res["cand_ids"] if any(g in chunks[i]["clauses"] for g in fq["gold"]))
    failure_results.append({**fq, "gold_chunk": gold_chunk, "baseline_rank": b_rank, "rerank_rank": r_rank,
                            "baseline_top3": [chunks[i]["clauses"][0] for i in res["baseline_top3"]],
                            "rerank_top3": [chunks[i]["clauses"][0] for i in res["rerank_top3"]],
                            "answer1": ans1, "answer2": ans2, "llm_s_1": t1, "llm_s_2": t2,
                            "table": rank_table(res, fq["gold"])})

    display(Markdown(f"### [{fq['category']}]  \n**Q:** {fq['query']}  \n"
                     f"**Gold clause(s):** {', '.join(fq['gold'])} → chunk {gold_chunk}  \n"
                     f"**Rank shift:** #{b_rank} (bi-encoder) → **#{r_rank}** (cross-encoder)  \n"
                     f"*Why it is hard:* {fq['why']}"))
    display(failure_results[-1]["table"])
    display(Markdown(f"**Answer 1 (baseline top-3: {failure_results[-1]['baseline_top3']}):**\n\n> {ans1}\n\n"
                     f"**Answer 2 (re-ranked top-3: {failure_results[-1]['rerank_top3']}):**\n\n> {ans2}\n\n---"))

### [Keyword trap / distractor]  
**Q:** Can a player use two hands to castle?  
**Gold clause(s):** 4.1, 7.5.4 → chunk 15  
**Rank shift:** #4 (bi-encoder) → **#1** (cross-encoder)  
*Why it is hard:* The word 'castle' pulls in the castling-procedure clauses (3.8.x, 4.4.x). The rule is in Art. 4.1 ('one hand only'), and the penalty is in 7.5.4, which never mentions castling as a topic.

,chunk,clauses,bi_rank,cosine,ce_rank,ce_score,shift,GOLD
0,38,"7.5.2, 7.5.3, 7.5.4",9,0.324,1,-0.09,8,★
1,98,"II.2.3, II.3, II.3.1",5,0.376,2,-0.18,3,
2,17,"4.4.2, 4.4.3, 4.4.4",1,0.454,3,-0.56,-2,
3,16,"4.3.2, 4.3.3, 4.4, 4.4.1",2,0.421,4,-3.54,-2,
4,15,"4.1, 4.2, 4.2.1, 4.2.2, 4.3",4,0.417,5,-4.66,-1,★
5,99,"II.3.2, II.3.2.1, II.3.2.2, II.3.2.3, II.3.2.4",6,0.359,6,-4.87,0,
6,102,II.3.2.7,10,0.321,7,-6.06,3,
7,12,"3.8.2.1, 3.8.2.2, 3.9",3,0.418,8,-6.50,-5,
8,26,"6.2.1.2, 6.2.2, 6.2.3",7,0.349,9,-8.19,-2,
9,84,"D.2.6.1, D.2.6.2",8,0.339,10,-8.75,-2,


**Answer 1 (baseline top-3: ['4.4.2', '4.3.2', '3.8.2.1']):**

> The provided context does not answer this.

**Answer 2 (re-ranked top-3: ['7.5.2', 'II.2.3', '4.4.2']):**

> No, according to Article 7.5.4, if a player uses two hands to make a single move (such as in case of castling) and presses the clock, it shall be considered and penalised as if an illegal move.

---

### [Keyword trap / synonyms]  
**Q:** Does a player have to keep writing down moves when very short of time?  
**Gold clause(s):** 8.4 → chunk 44  
**Rank shift:** #7 (bi-encoder) → **#1** (cross-encoder)  
*Why it is hard:* The query says 'writing down' and 'short of time'; Art. 8.4 says 'not obliged to meet the requirements of Article 8.1.1' and 'less than five minutes'. No shared vocabulary.

,chunk,clauses,bi_rank,cosine,ce_rank,ce_score,shift,GOLD
0,44,"8.4, 8.5, 8.5.1",7,0.496,1,2.86,6,★
1,89,"I.1, I.1.1",9,0.493,2,2.18,7,
2,65,"A.1, A.2, A.3, A.4",3,0.537,3,0.78,0,
3,104,"III.3.1.2, III.4",8,0.496,4,-1.33,4,
4,46,"8.6, 8.7",1,0.589,5,-1.56,-4,
5,121,Glossary,4,0.521,6,-2.73,-2,
6,45,"8.5.2, 8.5.3",10,0.490,7,-3.01,3,
7,42,"8.1.3, 8.1.4, 8.1.5",2,0.538,8,-3.15,-6,
8,41,"8.1, 8.1.1, 8.1.1.1, 8.1.1.2, 8.1.2",6,0.496,9,-4.35,-3,
9,66,"A.4.1, A.4.2, A.4.3, A.5, A.5.1",5,0.502,10,-5.61,-5,


...

,chunk,clauses,bi_rank,cosine,ce_rank,ce_score,shift,GOLD
0,3,"1.4.1, 1.4.2, 1.5",2,0.490,1,0.55,1,
1,22,"5.1.1, 5.1.2",1,0.519,2,-2.88,-1,
2,69,"A.5.3, A.5.4, A.5.5, A.6",8,0.417,3,-3.16,5,
3,105,"III.5, III.5.1",9,0.403,4,-4.56,5,
4,32,"6.9, 6.10, 6.10.1",3,0.447,5,-4.57,-2,★
5,23,"5.2.1, 5.2.2",4,0.422,6,-6.80,-2,
6,106,"III.5.2, III.5.3, III.6",10,0.392,7,-7.29,3,
7,118,Glossary,6,0.419,8,-7.96,-2,
8,31,"6.7.2, 6.8",5,0.419,9,-8.55,-4,
9,119,Glossary,7,0.418,10,-8.79,-3,


**Answer 1 (baseline top-3: ['5.1.1', '1.4.1', '6.9']):**

> The provided context does not answer this.

**Answer 2 (re-ranked top-3: ['1.4.1', '5.1.1', 'A.5.3']):**

> The provided context does not answer this.

---

### [Counter-example: re-ranking hurts (negation)]  
**Q:** Is the arbiter allowed to tell players how many moves have been made?  
**Gold clause(s):** 12.6 → chunk 62  
**Rank shift:** #2 (bi-encoder) → **#7** (cross-encoder)  
*Why it is hard:* 12.6 ('shall not indicate the number of moves completed') is rank 2 for the bi-encoder; the cross-encoder rewards passages that literally talk about counting moves (7.3, 9.3) and drops it to rank 7.

,chunk,clauses,bi_rank,cosine,ce_rank,ce_score,shift,GOLD
0,36,"7.3, 7.4, 7.4.1, 7.4.2, 7.4.3",9,0.551,1,5.34,8,
1,51,"9.3.1, 9.3.2, 9.4, 9.5",10,0.535,2,5.31,8,
2,46,"8.6, 8.7",6,0.590,3,5.18,3,
3,66,"A.4.1, A.4.2, A.4.3, A.5, A.5.1",3,0.642,4,4.91,-1,
4,34,"6.11.4, 6.12.1, 6.12.2",5,0.591,5,4.87,0,
5,60,"11.9, 11.10, 11.11, 11.12",1,0.682,6,3.39,-5,
6,62,"12.4, 12.5, 12.6",2,0.647,7,3.00,-5,★
7,39,7.5.5,7,0.575,8,2.64,-1,
8,56,"11.2.4, 11.3, 11.3.1, 11.3.2",8,0.573,9,1.15,-1,
9,70,"B.1, B.2, B.2.1, B.2.2, B.2.3",4,0.598,10,0.19,-6,


**Answer 1 (baseline top-3: ['11.9', '12.4', 'A.4.1']):**

> No, the arbiter shall refrain from informing a player that his/her opponent has completed a move. This is stated clearly in Article 12.6 of Passage 2.

**Answer 2 (re-ranked top-3: ['7.3', '9.3.1', '8.6']):**

> No, the arbiter is not explicitly allowed to tell players how many moves have been made according to the given context. The context does not provide information on the arbiter's role in informing players about the number of moves.

---

In [9]:
# Summary of rank shifts for the stress-test queries
shift_df = pd.DataFrame([{
    "category": r["category"], "query": r["query"], "gold": ", ".join(r["gold"]), "gold_chunk": r["gold_chunk"],
    "bi_rank": r["baseline_rank"], "ce_rank": r["rerank_rank"],
    "shift": f"Chunk {r['gold_chunk']}: #{r['baseline_rank']} → #{r['rerank_rank']}",
    "in_top3_baseline": r["baseline_rank"] <= 3, "in_top3_rerank": r["rerank_rank"] <= 3}
    for r in failure_results])
shift_df

,category,query,gold,gold_chunk,bi_rank,ce_rank,shift,in_top3_baseline,in_top3_rerank
0,Keyword trap / distractor,Can a player use two hands to castle?,"4.1, 7.5.4",15,4,1,Chunk 15: #4 → #1,False,True
1,Keyword trap / synonyms,Does a player have to keep writing down moves when very short of time?,8.4,44,7,1,Chunk 44: #7 → #1,False,True
2,Exception / corner case,"In a rapid game, what happens if nobody notices an illegal move?",A.5.2,68,4,1,Chunk 68: #4 → #1,False,True
3,Exception / penalty located elsewhere,What is the penalty for an incorrect draw claim in rapid chess?,A.3,65,6,2,Chunk 65: #6 → #2,False,True
4,Negation & constraints,Is a game drawn automatically without any claim after a long series of moves with no captures?,9.6.2,53,4,1,Chunk 53: #4 → #1,False,True
5,Other: vital chunk buried at rank 4-10,Is the time spent before pressing the clock counted against the player?,6.2.2,26,4,1,Chunk 26: #4 → #1,False,True
6,Counter-example: re-ranking hurts,My opponent ran out of time but I only have a bare king left. Do I win?,6.9,32,3,5,Chunk 32: #3 → #5,True,False
7,Counter-example: re-ranking hurts (negation),Is the arbiter allowed to tell players how many moves have been made?,12.6,62,2,7,Chunk 62: #2 → #7,True,False


## 7. Aggregate retrieval evaluation (labelled query set)

To check that the stress tests aren't cherry-picked, I score both pipelines on **all 61 labelled questions**
(each with its gold clause numbers) using:

* **Hit@1 / Hit@3**: is a gold chunk at rank 1 / in the top 3 (top 3 is what the LLM sees)?
* **MRR@10**: mean reciprocal rank of the first gold chunk within the 10 candidates.
* **Recall@10**: is the gold chunk in the candidate pool at all? (This is the ceiling for re-ranking.
  The cross-encoder can only reorder what the bi-encoder retrieved.)

In [10]:
EVAL_SET = [
 ("What happens if a player's mobile phone is found on them during the game?", ["11.3.2.2"]),
 ("What is the penalty for making an illegal move a second time?", ["7.5.5"]),
 ("In a rapid game, what happens if nobody notices an illegal move?", ["A.5.2"]),
 ("Can a player castle if the rook is under attack?", ["3.8.2.2"]),
 ("My opponent ran out of time but I only have a bare king left. Do I win?", ["6.9"]),
 ("If a player resigns but the opponent has only a king, what is the result?", ["5.1.2"]),
 ("Is a player allowed to leave the playing area while it is their turn to move?", ["11.2.3.2"]),
 ("Can the arbiter tell a player that they forgot to press their clock?", ["12.6"]),
 ("Can a spectator point out an illegal move to the players?", ["12.7"]),
 ("When is a player supposed to offer a draw?", ["9.1.2.1"]),
 ("Can a pawn be promoted to a second queen even if no queen has been captured?", ["3.7.3.4"]),
 ("A pawn reaches the last rank and the player presses the clock without swapping it. What happens?", ["7.5.2"]),
 ("What is the penalty for an incorrect draw claim in rapid chess?", ["A.3"]),
 ("How much extra time does the opponent get after a first illegal move in a blitz game?", ["A.3", "B.3"]),
 ("Is a game drawn automatically without any claim after a long series of moves with no captures?", ["9.6.2"]),
 ("Does a player have to keep writing down moves when very short of time?", ["8.4"]),
 ("Can a player castle in a rapid game if the king was set up on the wrong square?", ["A.5.1.2"]),
 ("Is smoking allowed in the playing area?", ["11.3.4"]),
 ("When can I straighten pieces on their squares without having to move them?", ["4.2.1"]),
 ("If I touch my rook first and then my king, can I still castle?", ["4.4.2"]),
 ("What happens if the board was placed the wrong way round at the start?", ["7.2.2"]),
 ("What happens if a player arrives late for the game?", ["6.7.1"]),
 ("When is a player permitted to stop the clocks?", ["6.11.2"]),
 ("Can a player look at their notebook with opening analysis during play?", ["11.3.1"]),
 ("Can the players agree a draw before any moves are made?", ["5.2.3"]),
 ("If the 75th move delivers checkmate, is the game a draw or a win?", ["9.6.2"]),
 ("Can the arbiter announce a flag fall in a rapid game?", ["A.5.5"]),
 ("Is a king allowed to capture the opponent's king?", ["1.4.1"]),
 ("What does the arbiter do if both kings are in check in a blitz game?", ["A.5.4"]),
 ("Is it permitted to castle through a square controlled by an enemy piece?", ["3.8.2.2"]),
 ("Who can a player tell if they notice an irregularity in another game?", ["12.7"]),
 ("Can a player claim a draw by repetition after pressing the clock?", ["9.2.2"]),
 ("Is it forbidden to keep my hand hovering over the clock?", ["6.2.3"]),
 ("What punishments can the arbiter give for bad behaviour?", ["12.9.1", "12.9.4"]),
 ("Can colours be swapped if the game started with the wrong colours?", ["7.3"]),
 ("Can a player who does not have the move adjust the pieces?", ["6.2.5", "4.2.1"]),
 ("Is a player allowed to write their moves on the scoresheet before playing them?", ["8.1.2"]),
 ("Can a player withdraw a draw claim once it has been made?", ["9.5.1"]),
 ("Can a draw offer be taken back?", ["9.1.2.1"]),
 ("Is a player allowed to keep a switched-off phone in their bag?", ["11.3.2.1"]),
 ("Can a claim be based only on the move count shown on a demonstration board?", ["6.12.2"]),
 ("Is the arbiter allowed to tell players how many moves have been made?", ["12.6"]),
 ("Can a pawn be promoted to a king?", ["3.7.3.3"]),
 ("Is the time spent before pressing the clock counted against the player?", ["6.2.2"]),
 ("Can a player appeal after signing the scoresheet?", ["11.10"]),
 ("Can you castle out of check?", ["3.8.2.2"]),
 ("Can the king castle if it moved earlier but went back to its original square?", ["3.8.2.1"]),
 ("What happens if my opponent keeps offering draws just to annoy me?", ["11.5"]),
 ("Is a player who has finished their game allowed to stand and watch other games?", ["11.4"]),
 ("What happens if a player refuses to have their bag inspected?", ["11.3.3"]),
 ("Can the clock settings be changed after ten moves in a rapid game?", ["A.5.1.1"]),
 ("Is it prohibited to analyse the position on another board during the game?", ["11.3.1"]),
 ("Which pieces is a pawn not allowed to promote to?", ["3.7.3.3"]),
 ("Is a player who is not on move allowed to leave the playing area without permission?", ["11.2.4", "11.2.3.2"]),
 ("Can a player use two hands to castle?", ["7.5.4", "4.1"]),
 ("Can a game be won on time if the opponent has insufficient material?", ["6.9"]),
 ("What happens if the player having the move touches an opponent's piece?", ["4.3.2"]),
 ("Is an en passant capture still possible two moves later?", ["3.7.3.2"]),
 ("What score does a player get for a game lost by forfeit?", ["10.1"]),
 ("What is the punishment for having a mobile phone ring in the hall?", ["11.3.2.2"]),
 ("Who is allowed to enter the playing area besides players and arbiters?", ["11.2.3.3"]),
]

rows = []
for q, gold in EVAL_SET:
    res = run_both_retrievals(q)
    rows.append({"query": q, "gold": ", ".join(gold),
                 "bi_rank": gold_rank(res["cand_ids"], gold), "ce_rank": gold_rank(res["rr_ids"], gold)})
eval_df = pd.DataFrame(rows)

def metrics(col):
    r = eval_df[col]
    return {"Hit@1": (r == 1).mean(), "Hit@3": (r <= 3).mean(),
            "MRR@10": r.apply(lambda x: 1 / x if pd.notna(x) else 0).mean(),
            "Recall@10": r.notna().mean()}

metrics_df = pd.DataFrame({"Pipeline 1 (bi-encoder)": metrics("bi_rank"),
                           "Pipeline 2 (re-ranked)": metrics("ce_rank")}).T.round(3)
print(f"{len(eval_df)} labelled queries")
display(metrics_df)

eval_df["effect"] = np.select(
    [(eval_df.bi_rank > 3) & (eval_df.ce_rank <= 3), (eval_df.bi_rank <= 3) & (eval_df.ce_rank > 3)],
    ["rescued into top-3", "pushed out of top-3"], default="no top-3 change")
display(eval_df["effect"].value_counts().to_frame("queries"))
eval_df[eval_df.effect != "no top-3 change"].sort_values("effect")

61 labelled queries


,Hit@1,Hit@3,MRR@10,Recall@10
Pipeline 1 (bi-encoder),0.525,0.836,0.688,0.984
Pipeline 2 (re-ranked),0.639,0.852,0.771,0.984


,queries
effect,
no top-3 change,50
rescued into top-3,6
pushed out of top-3,5


,query,gold,bi_rank,ce_rank,effect
4,My opponent ran out of time but I only have a bare king left. Do I win?,6.9,3.0,5.0,pushed out of top-3
8,Can a spectator point out an illegal move to the players?,12.7,3.0,5.0,pushed out of top-3
13,How much extra time does the opponent get after a first illegal move in a blitz game?,"A.3, B.3",3.0,8.0,pushed out of top-3
31,Can a player claim a draw by repetition after pressing the clock?,9.2.2,2.0,4.0,pushed out of top-3
41,Is the arbiter allowed to tell players how many moves have been made?,12.6,2.0,7.0,pushed out of top-3
2,"In a rapid game, what happens if nobody notices an illegal move?",A.5.2,4.0,1.0,rescued into top-3
12,What is the penalty for an incorrect draw claim in rapid chess?,A.3,6.0,2.0,rescued into top-3
14,Is a game drawn automatically without any claim after a long series of moves with no captures?,9.6.2,4.0,1.0,rescued into top-3
15,Does a player have to keep writing down moves when very short of time?,8.4,7.0,1.0,rescued into top-3
43,Is the time spent before pressing the clock counted against the player?,6.2.2,4.0,1.0,rescued into top-3


## 8. Latency profiling

I time each stage separately (after a warm-up), repeating every query `N_REPEATS` times:

* **Pipeline 1 retrieval** = query embedding + FAISS top-10 search
* **Pipeline 2 retrieval** = Pipeline 1 retrieval + cross-encoder scoring of 10 pairs + sort
* **LLM generation** is timed separately (it dominates end-to-end time and is the same for both pipelines)

In [11]:
N_REPEATS = 20
lat_queries = [q for q, _ in EVAL_SET]

for q in lat_queries[:5]:            # warm-up (kernel/JIT/cache effects)
    run_both_retrievals(q)

embed_t, search_t, ce_t = [], [], []
for _ in range(N_REPEATS):
    for q in lat_queries:
        t0 = time.perf_counter()
        qv = bi_encoder.encode([q], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        t1 = time.perf_counter()
        _, ids = index.search(qv, TOP_K_RETRIEVE)
        t2 = time.perf_counter()
        rerank(q, ids[0].tolist())
        t3 = time.perf_counter()
        embed_t.append(t1 - t0); search_t.append(t2 - t1); ce_t.append(t3 - t2)

embed_t, search_t, ce_t = map(lambda a: np.array(a) * 1000, (embed_t, search_t, ce_t))   # → ms
p1 = embed_t + search_t
p2 = p1 + ce_t

def stats(a):
    return {"mean_ms": a.mean(), "median_ms": np.median(a), "p95_ms": np.percentile(a, 95)}

latency_df = pd.DataFrame({
    "Query embedding (bi-encoder)":           stats(embed_t),
    "FAISS top-10 search":                    stats(search_t),
    "Cross-encoder: score 10 pairs + sort":    stats(ce_t),
    "PIPELINE 1 retrieval total":             stats(p1),
    "PIPELINE 2 retrieval total":             stats(p2),
}).T.round(3)
print(f"{len(p1)} timed runs per stage on {DEVICE.upper()}")
display(latency_df)
print(f"Re-ranking overhead: +{p2.mean() - p1.mean():.1f} ms per query "
      f"({p2.mean() / p1.mean():.1f}x the baseline retrieval time)")

llm_df = pd.DataFrame([{"query": r["query"][:60], "LLM s (Answer 1)": round(r["llm_s_1"], 1),
                        "LLM s (Answer 2)": round(r["llm_s_2"], 1)} for r in failure_results])
print(f"\nLLM generation (greedy, ≤{MAX_NEW_TOKENS} new tokens, {LLM_NAME} on {DEVICE.upper()}): "
      f"mean {llm_df.iloc[:, 1:].values.mean():.1f} s per answer")
llm_df

1220 timed runs per stage on CPU


,mean_ms,median_ms,p95_ms
Query embedding (bi-encoder),7.511,7.486,8.719
FAISS top-10 search,0.039,0.038,0.046
Cross-encoder: score 10 pairs + sort,96.654,95.446,110.154
PIPELINE 1 retrieval total,7.550,7.528,8.758
PIPELINE 2 retrieval total,104.204,103.156,117.884


Re-ranking overhead: +96.7 ms per query (13.8x the baseline retrieval time)

LLM generation (greedy, ≤220 new tokens, Qwen/Qwen2.5-3B-Instruct on CPU): mean 8.7 s per answer


,query,LLM s (Answer 1),LLM s (Answer 2)
0,Can a player use two hands to castle?,3.5,11.0
1,Does a player have to keep writing down moves when very shor,13.2,11.5
2,"In a rapid game, what happens if nobody notices an illegal m",10.1,11.1
3,What is the penalty for an incorrect draw claim in rapid che,14.7,6.5
4,Is a game drawn automatically without any claim after a long,9.6,9.9
5,Is the time spent before pressing the clock counted against,6.7,5.3
6,My opponent ran out of time but I only have a bare king left,3.5,3.6
7,Is the arbiter allowed to tell players how many moves have b,8.9,10.3


## 9. Scaling experiment: why not cross-encode the whole collection?

A bi-encoder pays the document-encoding cost **once, offline**. A query then costs one embedding plus a vector search,
which ANN indexes such as HNSW or IVF make sub-linear in N. A cross-encoder has nothing to pre-compute, because every
score depends on the (query, chunk) pair. It must run a full transformer forward pass over **every chunk for every query**,
so cost grows linearly with N. Below I measure both on this document and extrapolate.

In [12]:
q = "Is a game drawn automatically without any claim after a long series of moves with no captures?"
reps = 5

t0 = time.perf_counter()
for _ in range(reps):
    full_scores = cross_encoder.predict([(q, p) for p in passages], batch_size=32)
ce_full_ms = (time.perf_counter() - t0) / reps * 1000

t0 = time.perf_counter()
for _ in range(reps):
    baseline_retrieve(q)
bi_ms = (time.perf_counter() - t0) / reps * 1000

per_pair_ms = ce_full_ms / len(passages)
print(f"Cross-encoder over ALL {len(passages)} chunks: {ce_full_ms:.0f} ms  ({per_pair_ms:.2f} ms / pair)")
print(f"Bi-encoder retrieval over all chunks:          {bi_ms:.1f} ms")

# Does full cross-encoding even rank better here? Compare gold rank: bi top-10 rerank vs full-corpus CE.
full_order = np.argsort(-full_scores).tolist()
print("gold rank, full-corpus CE:", gold_rank(full_order, ["9.6.2"]),
      "| gold rank, top-10 rerank:", gold_rank(rerank(q, baseline_retrieve(q)[0])[0], ["9.6.2"]))

scale = pd.DataFrame({"corpus_chunks": [len(passages), 10_000, 1_000_000, 100_000_000]})
scale["cross_encoder_all (s/query)"] = (scale.corpus_chunks * per_pair_ms / 1000).round(2)
scale["two-stage: bi top-10 + CE (s/query)"] = ((bi_ms + 10 * per_pair_ms) / 1000)   # ANN search ~constant
scale["two-stage: bi top-10 + CE (s/query)"] = scale["two-stage: bi top-10 + CE (s/query)"].round(3)
scale["CE-all human time"] = pd.to_timedelta(scale["cross_encoder_all (s/query)"], unit="s").astype(str)
print("\nExtrapolation (linear in N for CE; ANN search assumed ~constant for the bi-encoder):")
scale

Cross-encoder over ALL 141 chunks: 1655 ms  (11.74 ms / pair)
Bi-encoder retrieval over all chunks:          8.1 ms
gold rank, full-corpus CE: 1 | gold rank, top-10 rerank: 1

Extrapolation (linear in N for CE; ANN search assumed ~constant for the bi-encoder):


,corpus_chunks,cross_encoder_all (s/query),two-stage: bi top-10 + CE (s/query),CE-all human time
0,141,1.66,0.125,0 days 00:00:01.660000
1,10000,117.39,0.125,0 days 00:01:57.390000
2,1000000,11738.72,0.125,0 days 03:15:38.720000
3,100000000,1173871.83,0.125,13 days 14:04:31.830000


In [13]:
# Re-rank depth trade-off: how many bi-encoder candidates should the cross-encoder see?
depth_rows = []
for depth in [3, 5, 10, 20, 50, len(passages)]:
    hits, times = [], []
    for qq, gold in EVAL_SET:
        t0 = time.perf_counter()
        cand, _ = baseline_retrieve(qq, k=depth)
        rr, _ = rerank(qq, cand)
        times.append(time.perf_counter() - t0)
        r = gold_rank(rr, gold)
        hits.append(r is not None and r <= 3)
    depth_rows.append({"re-rank depth (candidates)": depth, "Hit@3": round(np.mean(hits), 3),
                       "mean retrieval+rerank ms": round(np.mean(times) * 1000, 1)})
depth_df = pd.DataFrame(depth_rows)
depth_df

,re-rank depth (candidates),Hit@3,mean retrieval+rerank ms
0,3,0.836,40.3
1,5,0.885,59.1
2,10,0.852,105.1
3,20,0.852,194.1
4,50,0.852,450.5
5,141,0.852,1424.7


## 10. Save results (used for the report)

In [14]:
shift_df.to_csv("results/failure_rank_shifts.csv", index=False)
eval_df.to_csv("results/eval_per_query.csv", index=False)
metrics_df.to_csv("results/eval_metrics.csv")
latency_df.to_csv("results/latency_retrieval.csv")
llm_df.to_csv("results/latency_llm.csv", index=False)
scale.to_csv("results/scaling_extrapolation.csv", index=False)
depth_df.to_csv("results/rerank_depth.csv", index=False)

with open("results/failure_cases.json", "w", encoding="utf-8") as f:
    json.dump([{k: (v.to_dict(orient="records") if isinstance(v, pd.DataFrame) else v)
                for k, v in r.items()} for r in failure_results], f, indent=2, ensure_ascii=False)
print(sorted(os.listdir("results")))

['eval_metrics.csv', 'eval_per_query.csv', 'failure_cases.json', 'failure_rank_shifts.csv', 'latency_llm.csv', 'latency_retrieval.csv', 'rerank_depth.csv', 'scaling_extrapolation.csv']
